In [ ]:
# ── CELL 1: INSTALLATION ──────────────────────────
# Estimated time: 2-3 min. Wait for DONE.

import os, shutil, subprocess, sys

# 1. Install system dependencies
!apt-get install -y zstd curl lsof -q 2>/dev/null

# 2. Install Ollama
OLLAMA_PATHS = [
    shutil.which("ollama"),
    "/usr/local/bin/ollama",
    "/usr/bin/ollama",
    os.path.expanduser("~/.local/bin/ollama"),
    os.path.expanduser("~/ollama"),
]
if not any(p and os.path.exists(p) for p in OLLAMA_PATHS):
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("  Ollama already installed")

# 3. Install Open WebUI
try:
    import open_webui
    print("  Open WebUI already installed")
except ImportError:
    !pip install open-webui -q

# 4. Verify
missing = []
if not shutil.which("ollama"):
    ollama_found = any(p and os.path.exists(p) for p in OLLAMA_PATHS)
    if not ollama_found:
        missing.append("ollama")
try:
    import open_webui
except ImportError:
    missing.append("open-webui")

if missing:
    print(f"FAILED: {', '.join(missing)} not found")
else:
    print("=============================================")
    print("DONE - Proceed to Cell 2.")
    print("=============================================")


In [ ]:
# ── CELL 2: SETUP & RUN OLLAMA ─────────────────────
# Change model below if needed:
MODEL = "qwen2.5-coder:7b"   # default (~4.7GB)
# MODEL = "qwen2.5-coder:14b"  # coding (~9.0GB)
# MODEL = "deepseek-r1:7b"     # reasoning (~4.7GB)
# MODEL = "llama3.1:8b"        # general chat (~4.9GB)

import subprocess, time, os, shutil, json, urllib.request, sys

# --- Find ollama ---
OLLAMA_CANDIDATES = [
    shutil.which("ollama"),
    "/usr/local/bin/ollama",
    "/usr/bin/ollama",
    os.path.expanduser("~/.local/bin/ollama"),
    os.path.expanduser("~/ollama"),
]
ollama_bin = next((p for p in OLLAMA_CANDIDATES if p and os.path.exists(p)), None)
if not ollama_bin:
    print("Ollama not found. Run Cell 1 first.")
    sys.exit(1)

# --- Kill stale processes ---
os.system("pkill -f 'ollama serve' 2>/dev/null || true")
time.sleep(1)

# --- Start Ollama ---
print("[1/3] Starting Ollama...")
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
proc_ollama = subprocess.Popen(
    [ollama_bin, "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print("  Waiting for Ollama...", end="", flush=True)
ollama_ready = False
for _ in range(30):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        r.close()
        print(" OK")
        ollama_ready = True
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(2)
if not ollama_ready:
    print(" FAILED")
    sys.exit(1)

# --- Pull model ---
print("[2/3] Pulling model...")
pull = subprocess.Popen(
    [ollama_bin, "pull", MODEL],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in iter(pull.stdout.readline, ''):
    line = line.rstrip()
    if line:
        print(f"  {line}")
pull.wait()
if pull.returncode != 0:
    print(f"  Pull failed (exit {pull.returncode})")
    sys.exit(1)

# --- Mount Drive & backup ---
print("[3/3] Backing up to Drive...")
from google.colab import drive
if not os.path.exists("/content/drive"):
    drive.mount('/content/drive')
LOCAL_PATH = os.path.expanduser("~/.ollama/models")
DRIVE_PATH = "/content/drive/MyDrive/ollama_models"
if os.path.exists(LOCAL_PATH):
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for item in os.listdir(LOCAL_PATH):
        if item.startswith('.'):
            continue
        src = os.path.join(LOCAL_PATH, item)
        dst = os.path.join(DRIVE_PATH, item)
        if not os.path.exists(dst):
            try:
                if os.path.isdir(src):
                    shutil.copytree(src, dst, dirs_exist_ok=True)
                else:
                    shutil.copy2(src, dst)
            except Exception as e:
                print(f"    Backup failed: {e}")

print("=============================================")
print(f"MODEL {MODEL} READY - Proceed to Cell 3.")
print("=============================================")


In [ ]:
# ── CELL 3: WEBUI + PROXY + TUNNEL ──────────────────
# MUST KEEP RUNNING while using WebUI / opencode

import os, subprocess, time, urllib.request, re, shutil, sys, threading
from http.server import HTTPServer, BaseHTTPRequestHandler
import json

WEBUI_PORT = 8081
OLLAMA_PORT = 11434
PROXY_PORT = 8888

# --- Helper: kill process on a port ---
def kill_port(port):
    for cmd in [f"fuser -k {port}/tcp 2>/dev/null", f"lsof -ti:{port} | xargs kill -9 2>/dev/null"]:
        ret = os.system(cmd)
        if os.WIFEXITED(ret) and os.WEXITSTATUS(ret) == 0:
            return True
    return False

# ============================================================
# 1. START OPEN WEBUI
# ============================================================
webui_bin = shutil.which("open-webui") or shutil.which("open_webui")
if not webui_bin:
    print("open-webui not found. Run Cell 1 first.")
    raise SystemExit(1)

print("[1/5] Cleaning ports...", flush=True)
kill_port(WEBUI_PORT)
kill_port(PROXY_PORT)
time.sleep(1)

print("[2/5] Starting Open WebUI...", flush=True)
env = {**os.environ, "OLLAMA_BASE_URL": "http://127.0.0.1:11434", "WEBUI_AUTH": "False"}
proc_webui = subprocess.Popen(
    [webui_bin, "serve", "--port", str(WEBUI_PORT)],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print("  Waiting for WebUI...", flush=True)
for i in range(60):
    if proc_webui.poll() is not None:
        print(f"  WebUI failed (exit {proc_webui.returncode})")
        raise SystemExit(1)
    try:
        r = urllib.request.urlopen(f"http://127.0.0.1:{WEBUI_PORT}", timeout=5)
        r.close()
        print("  WebUI OK")
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(5)
else:
    print("  WebUI timeout")

# ============================================================
# 2. PYTHON PROXY -> Ollama
# ============================================================
class OllamaProxy(BaseHTTPRequestHandler):
    """Forwards /v1/* and /api/* to Ollama. No auth."""

    def _forward(self):
        body = None
        content_len = self.headers.get('Content-Length')
        if content_len:
            body = self.rfile.read(int(content_len))

        target = f"http://127.0.0.1:{OLLAMA_PORT}{self.path}"
        headers = {k: v for k, v in self.headers.items()
                   if k.lower() not in ('host', 'content-length', 'transfer-encoding')}

        req = urllib.request.Request(target, data=body, headers=headers, method=self.command)
        try:
            resp = urllib.request.urlopen(req, timeout=120)
            self.send_response(resp.status)
            for k, v in resp.headers.items():
                if k.lower() not in ('transfer-encoding', 'content-encoding'):
                    self.send_header(k, v)
            self.end_headers()
            self.wfile.write(resp.read())
        except urllib.error.HTTPError as e:
            self.send_response(e.code)
            self.end_headers()
            self.wfile.write(e.read())
        except Exception as e:
            self.send_response(502)
            self.end_headers()
            self.wfile.write(json.dumps({"error": str(e)}).encode())

    def do_GET(self): self._forward()
    def do_POST(self): self._forward()
    def do_DELETE(self): self._forward()
    def do_HEAD(self): self._forward()

    def log_message(self, fmt, *args):
        pass  # suppress logs

proxy_server = HTTPServer(('0.0.0.0', PROXY_PORT), OllamaProxy)
proxy_thread = threading.Thread(target=proxy_server.serve_forever, daemon=True)
proxy_thread.start()
print(f"[3/5] Python proxy running on port {PROXY_PORT} -> Ollama:{OLLAMA_PORT}")

# ============================================================
# 3. CLOUDFLARE TUNNEL -> Python proxy (port 8888)
# ============================================================
CLOUDFLARED_BIN = "/usr/local/bin/cloudflared"
if not os.path.exists(CLOUDFLARED_BIN):
    print("  Downloading cloudflared...", flush=True)
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED_BIN
    )
    os.chmod(CLOUDFLARED_BIN, 0o755)

os.system("pkill -f cloudflared 2>/dev/null || true")
time.sleep(1)

print("[4/5] Starting Cloudflare tunnel...", end="", flush=True)
proc_cf_ollama = subprocess.Popen(
    [CLOUDFLARED_BIN, "tunnel", "--url", f"http://127.0.0.1:{PROXY_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
ollama_public_url = None
for _ in range(30):
    line = proc_cf_ollama.stdout.readline()
    if not line: break
    m = re.search(r'(https://[a-zA-Z0-9.-]+\.trycloudflare\.com)', line)
    if m:
        ollama_public_url = m.group(1)
        print(" OK")
        break
    time.sleep(1)
    print(".", end="", flush=True)

if not ollama_public_url:
    print(" FAILED - tunnel URL not found")
    sys.exit(1)

# Also tunnel WebUI separately (for browser use)
print("  Starting Cloudflare tunnel for WebUI...", end="", flush=True)
proc_cf_webui = subprocess.Popen(
    [CLOUDFLARED_BIN, "tunnel", "--url", f"http://127.0.0.1:{WEBUI_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
webui_url = None
for _ in range(30):
    line = proc_cf_webui.stdout.readline()
    if not line: break
    m = re.search(r'(https://[a-zA-Z0-9.-]+\.trycloudflare\.com)', line)
    if m:
        webui_url = m.group(1)
        print(" OK")
        break
    time.sleep(1)
    print(".", end="", flush=True)

# ============================================================
# 4. TEST
# ============================================================
print("[5/5] Testing...")
try:
    r = urllib.request.urlopen(f"{ollama_public_url}/api/tags", timeout=15)
    data = json.loads(r.read())
    models = [m['name'] for m in data.get('models', [])]
    print(f"  Ollama accessible! Models: {models}")
except Exception as e:
    print(f"  Ollama test: {e}")

print()

# ============================================================
# 5. PRINT RESULTS
# ============================================================
SEP = "=" * 60
print(SEP)
print("WEBUI (open in browser):")
print(f"  {webui_url or 'FAILED'}")
print(SEP)
print()
print(SEP)
print("OLLAMA API (for opencode.json):")
print(f"  {ollama_public_url}")
print(SEP)
print()
print("Add this to opencode.json:")
print("{")
print('  "$schema": "https://opencode.ai/config.json",')
print('  "provider": {')
print('    "Ollama": {')
print('      "npm": "@ai-sdk/openai-compatible",')
print('      "name": "Ollama (Colab)",')
print('      "options": {')
print(f'        "baseURL": "{ollama_public_url}/v1"')
print('      },')
print('      "models": {')
print(f'        "{MODEL}": {{')
print(f'          "name": "{MODEL}"')
print('        }')
print('      }')
print('    }')
print('  }')
print("}")
print()
print("NOTES:")
print("  - Python proxy forwards /api/* and /v1/* to Ollama with NO auth")
print("  - Keep this cell running while using opencode")
print("  - If Colab disconnects, re-run Cell 1 -> 2 -> 3")
print(f"  - Model key: {MODEL}")
print()

# ============================================================
# 6. MONITOR
# ============================================================
print("Monitoring (Ctrl+C to stop)...")
try:
    while True:
        time.sleep(30)
        if proc_webui.poll() is not None:
            print("WebUI stopped!")
        if proc_cf_ollama.poll() is not None:
            print("Ollama tunnel stopped!")
            break
        print("All active -", time.strftime("%H:%M:%S"))
except KeyboardInterrupt:
    print("Stopped.")
